# SARGASSUM MVP — entrenamiento SATsum (P0)

Notebook ejecutable para cargar el Excel regional SATsum, construir features causales, evaluar un **baseline de persistencia**, entrenar un **XGBoost Regressor** por horizonte D+1…D+7 con backtesting temporal (sin filtración) y exportar artefactos a `outputs/`.

**Alcance:** biomasa satelital regional (producto oficial 2dr). **No** valida arribazón costera con este archivo solo.

**Requisitos:** Python 3.11/3.12 + `requirements.txt`. Solo Excel local; sin servicios externos.

## 0. Configuración y rutas

In [1]:
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

REPO_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_XLSX = REPO_ROOT / "data" / "raw" / "SATsum_diaria_serie_completa_2dr_zee_caribbean_2026-11-20_1632.xlsx"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
OUT_MODELS = REPO_ROOT / "outputs" / "models"
OUT_METRICS = REPO_ROOT / "outputs" / "metrics"
OUT_FIGURES = REPO_ROOT / "outputs" / "figures"
OUT_FORECASTS = REPO_ROOT / "outputs" / "forecasts"

for d in (PROCESSED_DIR, OUT_MODELS, OUT_METRICS, OUT_FIGURES, OUT_FORECASTS):
    d.mkdir(parents=True, exist_ok=True)

TARGET_COL_RAW = "2-day combined product · 2dr · OFFICIAL daily figure (t)"
TARGET_COL = "biomass_2dr_t"
HORIZONS = list(range(1, 8))
PURGE_DAYS = 7
TEST_DAYS = 365
VAL_DAYS = 365

assert RAW_XLSX.exists(), f"Missing workbook: {RAW_XLSX}"
print("REPO_ROOT:", REPO_ROOT)
print("RAW_XLSX:", RAW_XLSX)

REPO_ROOT: C:\Users\cyair\Documents\Dev\Personal\sargazo_Alert
RAW_XLSX: C:\Users\cyair\Documents\Dev\Personal\sargazo_Alert\data\raw\SATsum_diaria_serie_completa_2dr_zee_caribbean_2026-11-20_1632.xlsx


## 1. Carga y auditoría (hoja `Datos`)

Interpretación clave del producto 2dr: toma la observación del día o, si no hay observación válida, la del día anterior; **no es un promedio de dos días**. La cifra es regional (ZEE); no equivale a toneladas arribadas a playa.

In [2]:
df = pd.read_excel(RAW_XLSX, sheet_name="Datos")
try:
    meta = pd.read_excel(RAW_XLSX, sheet_name="Metadatos")
    print("Metadatos rows:", len(meta), "cols:", list(meta.columns)[:8])
except Exception as exc:  # noqa: BLE001
    meta = None
    print("Metadatos sheet not readable:", exc)

print("Datos shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Metadatos rows: 16 cols: ['Field', 'Value']
Datos shape: (6165, 8)
Columns: ['Date', 'Year', 'Month', 'Day', 'Day product · MODIS Aqua+Terra (t)', '2-day combined product · 2dr · OFFICIAL daily figure (t)', 'Area of the 2-day combined (km2)', 'Zone']


,Date,Year,Month,Day,Day product · MODIS Aqua+Terra (t),2-day combined product · 2dr · OFFICIAL daily figure (t),Area of the 2-day combined (km2),Zone
0,2010-01-01,2010,1,1,284,290,43,Mexican Caribbean EEZ
1,2010-01-02,2010,1,2,0,284,40,Mexican Caribbean EEZ
2,2010-01-03,2010,1,3,0,0,0,Mexican Caribbean EEZ
3,2010-01-04,2010,1,4,0,0,0,Mexican Caribbean EEZ
4,2010-01-05,2010,1,5,0,0,0,Mexican Caribbean EEZ


In [3]:
# Normalize schema
rename_map = {
    "Date": "date",
    TARGET_COL_RAW: TARGET_COL,
    "Day product · MODIS Aqua+Terra (t)": "biomass_modis_t",
    "Area of the 2-day combined (km2)": "area_2dr_km2",
    "Zone": "zone",
}
work = df.rename(columns=rename_map).copy()
work["date"] = pd.to_datetime(work["date"])
work = work.sort_values("date").reset_index(drop=True)

n_rows = len(work)
n_dup_dates = int(work["date"].duplicated().sum())
date_min, date_max = work["date"].min(), work["date"].max()
expected_days = pd.date_range(date_min, date_max, freq="D")
missing_days = expected_days.difference(work["date"])
negatives = int((work[TARGET_COL] < 0).sum()) if TARGET_COL in work else None
nulls = int(work[TARGET_COL].isna().sum()) if TARGET_COL in work else None
zeros = int((work[TARGET_COL] == 0).sum()) if TARGET_COL in work else None

audit = {
    "n_rows": n_rows,
    "n_columns": work.shape[1],
    "date_min": str(date_min.date()),
    "date_max": str(date_max.date()),
    "duplicate_dates": n_dup_dates,
    "missing_calendar_days": int(len(missing_days)),
    "target_nulls": nulls,
    "target_negatives": negatives,
    "target_zeros": zeros,
    "target_min": float(work[TARGET_COL].min()) if TARGET_COL in work else None,
    "target_max": float(work[TARGET_COL].max()) if TARGET_COL in work else None,
    "interpretation": {
        "target": "regional satellite 2dr official daily figure (t)",
        "not_coastal_arrival": True,
        "2dr_is_not_two_day_average": True,
    },
}
assert n_rows == 6165, f"Expected 6165 rows, got {n_rows}"
assert TARGET_COL in work.columns, f"Missing target column after rename: {TARGET_COL}"
assert n_dup_dates == 0, "Duplicate dates found; resolve before modeling"

(OUT_METRICS / "data_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
print(json.dumps(audit, indent=2))
if len(missing_days):
    print("First missing days:", [str(d.date()) for d in missing_days[:10]])

{
  "n_rows": 6165,
  "n_columns": 8,
  "date_min": "2010-01-01",
  "date_max": "2026-11-19",
  "duplicate_dates": 0,
  "missing_calendar_days": 2,
  "target_nulls": 0,
  "target_negatives": 0,
  "target_zeros": 1112,
  "target_min": 0.0,
  "target_max": 207566.0,
  "interpretation": {
    "target": "regional satellite 2dr official daily figure (t)",
    "not_coastal_arrival": true,
    "2dr_is_not_two_day_average": true
  }
}
First missing days: ['2021-04-25', '2021-08-16']


In [4]:
# Optional processed snapshot (immutable raw Excel stays untouched)
parquet_path = PROCESSED_DIR / "satsum_datos_normalized.parquet"
try:
    work.to_parquet(parquet_path, index=False)
    print("Wrote", parquet_path)
except Exception as exc:  # noqa: BLE001
    csv_path = PROCESSED_DIR / "satsum_datos_normalized.csv"
    work.to_csv(csv_path, index=False)
    print("Parquet unavailable (", exc, "); wrote", csv_path)

Wrote C:\Users\cyair\Documents\Dev\Personal\sargazo_Alert\data\processed\satsum_datos_normalized.parquet


## 2. Features causales y objetivos D+1…D+7

Solo información conocida en la fecha de emisión. Lags y rolling stats usan el pasado; estacionalidad con seno/coseno del día del año. Targets con `shift(-h)`.

In [5]:
feat = work[["date", TARGET_COL]].copy()
y = feat[TARGET_COL].astype(float)

# Lags
for lag in (1, 2, 7, 14):
    feat[f"lag_{lag}"] = y.shift(lag)

# Rolling stats (shifted so the window ends at t-1)
for w in (7, 14, 30):
    feat[f"rolling_mean_{w}"] = y.shift(1).rolling(w).mean()
feat["rolling_std_7"] = y.shift(1).rolling(7).std()
feat["diff_1"] = y.diff(1)
feat["pct_change_7"] = y.pct_change(7).replace([np.inf, -np.inf], np.nan)

doy = feat["date"].dt.dayofyear
feat["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
feat["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

for h in HORIZONS:
    feat[f"target_h{h}"] = y.shift(-h)

FEATURE_COLS = [
    c
    for c in feat.columns
    if c.startswith(("lag_", "rolling_", "diff_", "pct_", "doy_"))
]
print("Feature columns:", FEATURE_COLS)
feat.tail(10)

Feature columns: ['lag_1', 'lag_2', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30', 'rolling_std_7', 'diff_1', 'pct_change_7', 'doy_sin', 'doy_cos']


C:\Users\cyair\AppData\Local\Temp\ipykernel_21892\208524830.py:15: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  doy = feat["date"].dt.dayofyear


,date,biomass_2dr_t,lag_1,lag_2,lag_7,lag_14,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_std_7,...,pct_change_7,doy_sin,doy_cos,target_h1,target_h2,target_h3,target_h4,target_h5,target_h6,target_h7
6155,2026-11-10,86,625.0,1820.0,7373.0,3415.0,2815.714286,4877.714286,4262.766667,2201.890227,...,-0.988336,-0.771773,0.635898,694.0,837.0,415.0,185.0,496.0,540.0,44.0
6156,2026-11-11,694,86.0,625.0,3607.0,1583.0,1774.714286,4639.928571,3836.300000,1168.092707,...,-0.807596,-0.760720,0.649080,837.0,415.0,185.0,496.0,540.0,44.0,21.0
6157,2026-11-12,837,694.0,86.0,2216.0,4854.0,1358.571429,4576.428571,3790.666667,893.037113,...,-0.622292,-0.749442,0.662070,415.0,185.0,496.0,540.0,44.0,21.0,126.0
6158,2026-11-13,415,837.0,694.0,2437.0,4772.0,1161.571429,4289.500000,3818.533333,821.612410,...,-0.829709,-0.737943,0.674863,185.0,496.0,540.0,44.0,21.0,126.0,NaN
6159,2026-11-14,185,415.0,837.0,1632.0,16141.0,872.714286,3978.285714,3767.033333,632.042117,...,-0.886642,-0.726225,0.687457,496.0,540.0,44.0,21.0,126.0,NaN,NaN
6160,2026-11-15,496,185.0,415.0,1820.0,11942.0,666.000000,2838.571429,3656.233333,576.510191,...,-0.727473,-0.714292,0.699848,540.0,44.0,21.0,126.0,NaN,NaN,NaN
6161,2026-11-16,540,496.0,185.0,625.0,5871.0,476.857143,2021.000000,3621.100000,271.091269,...,-0.136000,-0.702148,0.712031,44.0,21.0,126.0,NaN,NaN,NaN,NaN
6162,2026-11-17,44,540.0,496.0,86.0,7373.0,464.714286,1640.214286,3639.100000,265.189061,...,-0.488372,-0.689796,0.724004,21.0,126.0,NaN,NaN,NaN,NaN,NaN
6163,2026-11-18,21,44.0,540.0,694.0,3607.0,458.714286,1116.714286,3225.133333,275.461863,...,-0.969741,-0.677240,0.735762,126.0,NaN,NaN,NaN,NaN,NaN,NaN
6164,2026-11-19,126,21.0,44.0,837.0,2216.0,362.571429,860.571429,2987.466667,296.312255,...,-0.849462,-0.664483,0.747303,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Anti-leakage spot check: no feature at t may depend on biomass after t
# (lags/rolling use shift; targets are separate columns)
assert all(not c.startswith("target_") for c in FEATURE_COLS)

# Target alignment check for h=1 on a known row
sample_idx = 100
t0 = feat.loc[sample_idx, "date"]
y_h1 = feat.loc[sample_idx, "target_h1"]
y_next = feat.loc[sample_idx + 1, TARGET_COL]
assert pd.isna(y_h1) or np.isclose(y_h1, y_next), "target_h1 misaligned"
print("Leakage/target checks OK for sample", t0.date())

Leakage/target checks OK for sample 2010-04-11


## 3. Split temporal + baseline de persistencia

Corte cronológico. Purga de al menos 7 días entre el último origen de entrenamiento y el inicio de evaluación. El conjunto de prueba final permanece intacto (sin tuning sobre él).

In [7]:
max_date = feat["date"].max()
test_start = max_date - pd.Timedelta(days=TEST_DAYS - 1)
val_end = test_start - pd.Timedelta(days=PURGE_DAYS + 1)
val_start = val_end - pd.Timedelta(days=VAL_DAYS - 1)
train_end = val_start - pd.Timedelta(days=PURGE_DAYS + 1)

split_info = {
    "train_end": str(train_end.date()),
    "val_start": str(val_start.date()),
    "val_end": str(val_end.date()),
    "test_start": str(test_start.date()),
    "test_end": str(max_date.date()),
    "purge_days": PURGE_DAYS,
}
print(json.dumps(split_info, indent=2))


def mask_origin(frame: pd.DataFrame, start=None, end=None) -> pd.Series:
    m = pd.Series(True, index=frame.index)
    if start is not None:
        m &= frame["date"] >= start
    if end is not None:
        m &= frame["date"] <= end
    return m


def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_pred = np.asarray(y_pred, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
        "n": int(len(y_true)),
    }


baseline_rows = []
for h in HORIZONS:
    target = f"target_h{h}"
    # Persistence: predict y_t for y_{t+h}
    tmp = feat.loc[mask_origin(feat, test_start, max_date), ["date", TARGET_COL, target]].dropna()
    # Origin dates in test; target already known historically for backtest
    y_true = tmp[target].to_numpy()
    y_pred = tmp[TARGET_COL].to_numpy()  # last known value at origin date t
    metrics = evaluate_predictions(y_true, y_pred)
    metrics.update({"model": "persistence", "horizon": h})
    baseline_rows.append(metrics)

baseline_df = pd.DataFrame(baseline_rows)
baseline_df

{
  "train_end": "2024-11-05",
  "val_start": "2024-11-13",
  "val_end": "2025-11-12",
  "test_start": "2025-11-20",
  "test_end": "2026-11-19",
  "purge_days": 7
}


,mae,rmse,r2,n,model,horizon
0,10197.510989,22692.621778,0.412012,364,persistence,1
1,16456.658402,31355.440206,-0.120376,363,persistence,2
2,17976.914365,34682.041165,-0.367817,362,persistence,3
3,18428.476454,36912.583470,-0.546156,361,persistence,4
4,16908.608333,35573.360834,-0.433169,360,persistence,5
5,18664.331476,37177.600367,-0.562235,359,persistence,6
6,19098.371508,37051.457366,-0.547795,358,persistence,7


## 4. XGBoost Regressor por horizonte

Entrenamiento en CPU con hiperparámetros fijos modestos (validación temporal interna en el bloque val; **sin** tocar el test para selección). Salidas truncadas a >= 0.

In [8]:
xgb_params = dict(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=RANDOM_SEED,
    n_jobs=2,
)

models = {}
xgb_rows = []
val_rows = []
pred_frames = []

t0 = time.perf_counter()
for h in HORIZONS:
    target = f"target_h{h}"
    cols = FEATURE_COLS + [target, "date", TARGET_COL]
    data_h = feat[cols].dropna().copy()

    train_m = mask_origin(data_h, end=train_end)
    val_m = mask_origin(data_h, val_start, val_end)
    test_m = mask_origin(data_h, test_start, max_date)

    X_train, y_train = data_h.loc[train_m, FEATURE_COLS], data_h.loc[train_m, target]
    X_val, y_val = data_h.loc[val_m, FEATURE_COLS], data_h.loc[val_m, target]
    X_test, y_test = data_h.loc[test_m, FEATURE_COLS], data_h.loc[test_m, target]

    # Direct vs log1p: pick by validation MAE (test untouched)
    candidates = {}
    for name, transform in (("direct", False), ("log1p", True)):
        model = XGBRegressor(**xgb_params)
        if transform:
            model.fit(X_train, np.log1p(y_train.clip(lower=0)))
            pred_val = np.expm1(model.predict(X_val))
        else:
            model.fit(X_train, y_train)
            pred_val = model.predict(X_val)
        pred_val = np.clip(pred_val, 0, None)
        mae_val = float(mean_absolute_error(y_val, pred_val))
        candidates[name] = (model, transform, mae_val)

    best_name = min(candidates, key=lambda k: candidates[k][2])
    best_model, use_log, val_mae = candidates[best_name]

    # Refit on train+val (still before purge/test) for final test eval
    trainval_m = train_m | val_m
    X_trv, y_trv = data_h.loc[trainval_m, FEATURE_COLS], data_h.loc[trainval_m, target]
    final_model = XGBRegressor(**xgb_params)
    if use_log:
        final_model.fit(X_trv, np.log1p(y_trv.clip(lower=0)))
        pred_test = np.expm1(final_model.predict(X_test))
    else:
        final_model.fit(X_trv, y_trv)
        pred_test = final_model.predict(X_test)
    pred_test = np.clip(pred_test, 0, None)

    metrics = evaluate_predictions(y_test.to_numpy(), pred_test)
    metrics.update(
        {
            "model": "xgboost",
            "horizon": h,
            "transform": best_name,
            "val_mae": val_mae,
        }
    )
    xgb_rows.append(metrics)
    val_rows.append({"horizon": h, "chosen_transform": best_name, "val_mae": val_mae})

    models[h] = {"model": final_model, "transform": best_name, "features": FEATURE_COLS}
    joblib.dump(models[h], OUT_MODELS / f"xgb_regressor_h{h}.joblib")

    pred_frames.append(
        pd.DataFrame(
            {
                "origin_date": data_h.loc[test_m, "date"].to_numpy(),
                "horizon": h,
                "y_true": y_test.to_numpy(),
                "y_pred_xgb": pred_test,
                "y_pred_persistence": data_h.loc[test_m, TARGET_COL].to_numpy(),
            }
        )
    )
    print(f"h={h}: transform={best_name} val_mae={val_mae:.3f} test_mae={metrics['mae']:.3f}")

elapsed = time.perf_counter() - t0
print(f"Training wall time: {elapsed:.1f}s")

xgb_df = pd.DataFrame(xgb_rows)
comparison = (
    baseline_df[["horizon", "mae", "rmse"]]
    .rename(columns={"mae": "mae_persistence", "rmse": "rmse_persistence"})
    .merge(
        xgb_df[["horizon", "mae", "rmse", "r2", "transform"]].rename(
            columns={"mae": "mae_xgb", "rmse": "rmse_xgb", "r2": "r2_xgb"}
        ),
        on="horizon",
    )
)
comparison["mae_lift_vs_persistence"] = comparison["mae_persistence"] - comparison["mae_xgb"]
comparison

h=1: transform=direct val_mae=6483.872 test_mae=11581.405


h=2: transform=direct val_mae=8639.176 test_mae=14333.190


h=3: transform=direct val_mae=9171.436 test_mae=14823.415


h=4: transform=direct val_mae=8672.523 test_mae=15082.528


h=5: transform=direct val_mae=8683.369 test_mae=15129.097


h=6: transform=direct val_mae=9102.124 test_mae=15303.029


h=7: transform=direct val_mae=8636.073 test_mae=15743.032
Training wall time: 4.4s


,horizon,mae_persistence,rmse_persistence,mae_xgb,rmse_xgb,r2_xgb,transform,mae_lift_vs_persistence
0,1,10197.510989,22692.621778,11581.405354,21524.966994,0.458568,direct,-1383.894365
1,2,16456.658402,31355.440206,14333.190022,26691.518520,0.175571,direct,2123.468380
2,3,17976.914365,34682.041165,14823.414837,27539.125526,0.174771,direct,3153.499528
3,4,18428.476454,36912.583470,15082.527705,29283.056833,0.060219,direct,3345.948749
4,5,16908.608333,35573.360834,15129.097361,28150.401954,0.141707,direct,1779.510973
5,6,18664.331476,37177.600367,15303.028782,28363.041680,0.135071,direct,3361.302694
6,7,19098.371508,37051.457366,15743.032380,29432.316005,0.070998,direct,3355.339128


In [9]:
# Persist metrics
comparison.to_csv(OUT_METRICS / "horizon_comparison.csv", index=False)
metrics_payload = {
    "split": split_info,
    "baseline": baseline_rows,
    "xgboost": xgb_rows,
    "feature_columns": FEATURE_COLS,
    "xgb_params": xgb_params,
    "training_seconds": elapsed,
    "random_seed": RANDOM_SEED,
    "scope_warning": (
        "Regional satellite biomass only. Coastal beach arrival cannot be validated "
        "from this workbook alone."
    ),
}
(OUT_METRICS / "metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
pd.DataFrame(val_rows).to_csv(OUT_METRICS / "val_transform_choice.csv", index=False)
print("Wrote metrics to", OUT_METRICS)

Wrote metrics to C:\Users\cyair\Documents\Dev\Personal\sargazo_Alert\outputs\metrics


## 5. Figuras y predicción demostrativa

In [10]:
preds = pd.concat(pred_frames, ignore_index=True)
preds.to_csv(OUT_FORECASTS / "backtest_predictions_test.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(comparison["horizon"], comparison["mae_persistence"], marker="o", label="Persistence")
axes[0].plot(comparison["horizon"], comparison["mae_xgb"], marker="o", label="XGBoost")
axes[0].set_xlabel("Horizon (days)")
axes[0].set_ylabel("MAE (t)")
axes[0].set_title("Test MAE by horizon")
axes[0].legend()

h1 = preds[preds["horizon"] == 1]
axes[1].plot(h1["origin_date"], h1["y_true"], label="Observed", linewidth=1)
axes[1].plot(h1["origin_date"], h1["y_pred_xgb"], label="XGBoost", linewidth=1, alpha=0.8)
axes[1].set_title("D+1 backtest (test window)")
axes[1].set_ylabel("biomass_2dr_t")
axes[1].legend()
fig.autofmt_xdate()
fig.tight_layout()
fig_path = OUT_FIGURES / "mae_and_d1_backtest.png"
fig.savefig(fig_path, dpi=140)
plt.show()
print("Wrote", fig_path)

Wrote C:\Users\cyair\Documents\Dev\Personal\sargazo_Alert\outputs\figures\mae_and_d1_backtest.png


C:\Users\cyair\AppData\Local\Temp\ipykernel_21892\1444520968.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Demo operational-style forecast from the last origin date with full features
last_row = feat.dropna(subset=FEATURE_COLS).iloc[-1]
origin = last_row["date"]
demo = []
for h in HORIZONS:
    bundle = models[h]
    x = last_row[FEATURE_COLS].astype(float).to_frame().T
    pred = float(bundle["model"].predict(x)[0])
    if bundle["transform"] == "log1p":
        pred = float(np.expm1(pred))
    pred = float(max(pred, 0.0))
    demo.append(
        {
            "origin_date": str(origin.date()),
            "horizon": h,
            "target_date": str((origin + pd.Timedelta(days=h)).date()),
            "prediction_biomass_2dr_t": pred,
            "model": f"xgb_regressor_h{h}",
            "warning": (
                "Regional SATsum 2dr estimate only; not a validated coastal arrival forecast."
            ),
        }
    )

demo_df = pd.DataFrame(demo)
demo_path = OUT_FORECASTS / "demo_forecast_d1_d7.csv"
demo_df.to_csv(demo_path, index=False)
demo_df

,origin_date,horizon,target_date,prediction_biomass_2dr_t,model,warning
0,2026-11-19,1,2026-11-20,508.029968,xgb_regressor_h1,Regional SATsum 2dr estimate only; not a valid...
1,2026-11-19,2,2026-11-21,389.810974,xgb_regressor_h2,Regional SATsum 2dr estimate only; not a valid...
2,2026-11-19,3,2026-11-22,510.933807,xgb_regressor_h3,Regional SATsum 2dr estimate only; not a valid...
3,2026-11-19,4,2026-11-23,516.911377,xgb_regressor_h4,Regional SATsum 2dr estimate only; not a valid...
4,2026-11-19,5,2026-11-24,1622.266724,xgb_regressor_h5,Regional SATsum 2dr estimate only; not a valid...
5,2026-11-19,6,2026-11-25,0.000000,xgb_regressor_h6,Regional SATsum 2dr estimate only; not a valid...
6,2026-11-19,7,2026-11-26,1888.799927,xgb_regressor_h7,Regional SATsum 2dr estimate only; not a valid...


## 6. Notas de aceptación

- T16: este notebook no importa drivers de BD ni lee variables de conexión.
- T17: métricas y gráficos se refieren a biomasa regional 2dr, no a arribazón en playa.
- Si XGBoost no supera al baseline en algún horizonte, queda documentado en `outputs/metrics/horizon_comparison.csv` (no ocultar el resultado).

**Definition of Done (P0):** Excel leído, baseline + XGBoost, siete horizontes sin leakage, artefactos en `outputs/`, reproducible en laptop solo con el Excel y las dependencias locales.